# Day 7 | ILT 1: Gold Layer Build Strategy — Assembling the Star Schema
### GlobalMart Data Engineering Bootcamp

| | |
|---|---|
| **Format** | Instructor-led — run live, no student setup needed |
| **Builds on** | Day 6 (dimensions), Day 7 HOL 1 (bridge table concept) |
| **Leads into** | Day 7 HOL 2 — the actual `fact_sales` build |
| **Duration** | 90 minutes |

### Learning Objectives
- Understand the correct *order* in which a Gold layer gets assembled, and why that order isn't arbitrary
- Walk the exact join chain `fact_sales` needs, table by table
- Understand the natural-key vs surrogate-key join decision — and why GlobalMart's real build deliberately chose one over the other
- Preview `OVERWRITE` vs incremental refresh strategy for Gold (full detail comes Day 9–10)

This notebook runs entirely against the real `gbmart` catalog — every cell below is safe to run live in class exactly as written.

---
## 1. Why Order Matters: Dimensions Before Facts

A star schema can only be assembled in one direction: **dimensions first, fact second.** The fact table's whole job is to look up context by joining *out* to dimensions — if `dim_product` doesn't exist yet, there's nothing for `fact_sales` to join against when it needs `Actual_price`/`Discounted_price`.

```
Day 6:  Silver  ──▶  dim_customer, dim_product, dim_date, dim_address, dim_payment_method, dim_orders
Day 7:  Silver + Gold dims  ──▶  fact_sales
```

This is why Day 6 explicitly stopped before building `fact_sales` — it wasn't a scheduling accident, it's the only order that works.

In [ ]:
# ============================================================
# CELL 1: Confirm every dimension fact_sales will need actually exists
# before we start designing the fact table's build strategy.
# ============================================================

for table in ["dim_customer", "dim_product", "dim_date", "dim_address", "dim_payment_method"]:
    exists = spark.catalog.tableExists(f"gbmart.gold.{table}")
    print(f"gbmart.gold.{table:<22} exists: {exists}")

---
## 2. The Join Chain, Table by Table

`fact_sales`'s grain is `order_items` (one row per order line). Everything else is a lookup *from* that grain. This is the exact chain GlobalMart's real Gold build uses:

| Step | Join | Brings in |
|---|---|---|
| 1 | `silver.order_items` (the grain itself) | `order_item_id`, `order_id`, `product_id`, `quantity` |
| 2 | + `silver.orders` on `order_id` | `customer_id`, `order_date` |
| 3 | + `silver.products` (`is_current = true`) on `product_id` | `Actual_price`, `Discounted_price` |
| 4 | + `gold.dim_date` on `order_date = date` | `Time_ID` (= `date_key`) |
| 5 | + `silver.address` (one row per customer — see below) on `customer_id` | `Address_ID` |
| 6 | + `silver.payments` on `order_id` | `Payment_ID` |

**Why `silver.products` needs `is_current = true`:** `dim_product` is SCD2 — a product can have more than one row (one per price version) in Silver. Filtering to the current version before joining is what keeps this a clean many-to-one join instead of accidentally fanning out order lines.

---
## 3. The One-to-Many Trap: Address

`silver.address` links to `customer_id`, not to a specific order — a customer can have more than one address on file (billing, shipping, or both). Joining `order_items` straight to `silver.address` on `customer_id` would silently **fan out every order line** for any customer with 2+ addresses.

**The fix: pick exactly one address per customer before joining**, using a window function ranked so a `Shipping` address wins if one exists, otherwise take whatever's first:

In [ ]:
# ============================================================
# CELL 2: Live demo of the fan-out trap and its fix
# ============================================================

from pyspark.sql.functions import col, when, row_number
from pyspark.sql.window import Window

multi_address_customers = (
    spark.table("gbmart.silver.address")
    .groupBy("customer_id").count()
    .filter("count > 1")
)
print(f"Customers with more than one address on file: {multi_address_customers.count():,}")
print("Every one of these would silently duplicate order lines without the fix below.")

# The fix: rank addresses per customer, Shipping wins, take rank 1 only
address_window = Window.partitionBy("customer_id").orderBy(
    when(col("address_type").contains("Shipping"), 0).otherwise(1)
)
address_primary = (
    spark.table("gbmart.silver.address")
    .withColumn("_rank", row_number().over(address_window))
    .filter("_rank = 1")
)
print(f"Rows after picking one address per customer: {address_primary.count():,}")
print(f"Distinct customers with an address: {address_primary.select('customer_id').distinct().count():,}")
print("These two numbers should match -- exactly one address per customer, guaranteed.")

---
## 4. The Big Decision: Natural Keys or Surrogate Keys in the Fact Table?

Every dimension you built on Day 6 has a proper surrogate key (`customer_sk`, `product_sk`, ...). The textbook-Kimball answer is: `fact_sales` should join to dimensions **by surrogate key**, not by natural/business key — that's what makes SCD2 history actually queryable ("what did this order look like using the customer's address *at the time*").

**GlobalMart's real, running build makes a different, deliberate choice**: `fact_sales` stores `Customer_ID`, `Product_ID`, `Order_ID`, `Address_ID`, `Payment_ID` — natural keys, not surrogate keys. Only `fact_sales_sk` itself (the fact's own row identifier) is generated.

| | Surrogate-key fact (textbook) | Natural-key fact (GlobalMart's real build) |
|---|---|---|
| Point-in-time accuracy | Yes — join lands on the exact SCD2 version active at order time | No — always resolves to whatever is "current" when you query |
| Join complexity | Higher — need effective-date range logic at load time | Lower — plain equi-join on IDs students already know |
| Build/debug speed for a training program | Slower | Faster — this is the reason it was chosen here |
| Production-grade for a real BI team | Preferred | Acceptable as a documented simplification |

**This is not a mistake to "fix" in HOL 2 — build it exactly this way.** Understanding *why* a real team would make this tradeoff, and being able to say what it costs you, is the actual lesson. You'll build `fact_sales` with natural keys next, faithfully matching the real system.

---
## 5. Write Strategy: Why `overwrite` for Today, and What Changes Later

Today's `fact_sales` build (HOL 2) uses `mode("overwrite")` — same as every dimension in Day 6. At GlobalMart's current data volume this is simple and always correct: re-run the notebook, get a fresh, fully-accurate `fact_sales`.

This does **not** scale forever — recomputing the entire fact table from scratch every time new orders land becomes wasteful as volume grows. Day 9–10 replaces this with an **incremental `MERGE`-based refresh** that only touches the new/changed order lines. You're seeing the simple version first on purpose — the incremental version will make a lot more sense once you've felt why a full rebuild is wasteful.

---
## Recap Before HOL 2

| Decision | GlobalMart's choice |
|---|---|
| Build order | Dimensions (Day 6) fully before the fact (today) |
| Grain | One row per order line item (`order_items`) |
| Address fan-out | Resolved with a ranked window function — one address per customer |
| Key strategy | Natural keys in the fact, surrogate keys stay in the dimensions |
| Write strategy (today) | Full `overwrite` — incremental `MERGE` comes Day 9–10 |

Next: HOL 2 builds this for real.